First, ensure there's adversarial_results folder. If not, run adversarial_eval.py to generate the raw data for these plots. Then run this notebook to generate the plots.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import urllib.request

urllib.request.urlretrieve(
    'https://github.com/google/fonts/raw/main/ofl/ibmplexmono/IBMPlexMono-Regular.ttf',
    'IBMPlexMono-Regular.ttf'
)
fe = font_manager.FontEntry(fname='IBMPlexMono-Regular.ttf', name='plexmono')
font_manager.fontManager.ttflist.append(fe)

plt.rcParams.update({
    'axes.facecolor': '#f5f4e9',
    'grid.color': '#AAAAAA',
    'axes.edgecolor': '#333333',
    'figure.facecolor': '#FFFFFF',
    'axes.grid': False,
    'axes.prop_cycle': plt.cycler('color', plt.cm.Dark2.colors),
    'font.family': fe.name,
    'figure.figsize': (3.5, 3.5 / 1.2),
    'ytick.left': True,
    'xtick.bottom': True,
    'figure.dpi': 300,
})
os.makedirs('paper_figs', exist_ok=True)

In [ ]:
SUMMARY_CSV   = '../adversarial_results/adversarial_summary.csv'
ALL_RESULTS   = '../../gcn_results/all_results'
RUN_DATE      = '2026-03-14'
CENSOR_REGION = 'above'

df = pd.read_csv(SUMMARY_CSV)
CENSOR_SPLITS = sorted(df['censor_split'].unique())
NOISE_TYPES   = ['omission', 'xnoise','ynoise']

print(f'{len(df)} rows | censor_splits: {CENSOR_SPLITS}')
df.head()

In [ ]:
LABELS = {
    'ynoise':   'Label noise',
    'xnoise':   'Feature noise',
    'omission': 'Omission',
}
NT_COLORS = {'omission': 'C6', 'xnoise': 'C2', 'ynoise': 'C3'}
NT_MAP = {'ynoise': 'ynoise', 'xnoise': 'xnoise', 'omission': 'omit'}
NT_MARKERS = {'omission': 'o', 'xnoise': 's', 'ynoise': '^'}


def xnoise_to_level(s):
    a, b = map(float, str(s).split('-'))
    return round(1 - (a + b) / 2, 3)


def noise_param_to_level(noise_type, noise_param):
    if noise_type == 'xnoise':
        return xnoise_to_level(noise_param)
    return float(noise_param)


def highest_noise_param(df, noise_type):
    params = df[df['noise_type'] == noise_type]['noise_param'].unique()
    return max(params, key=lambda p: noise_param_to_level(noise_type, p))


def recovery_curve(df, noise_type, split, metric='upper_corr'):
    """Highest noise level for noise_type. Returns (clean_frac, y, yerr)."""
    param = highest_noise_param(df, noise_type)
    sub = df[
        (df['noise_type'] == noise_type) &
        (df['censor_split'] == split) &
        (df['noise_param'] == param)
    ]
    grp = sub.groupby('clean_frac').agg(
        nm=(f'noised_{metric}_mean',    'mean'),
        ns=(f'noised_{metric}_std',     'mean'),
        fm=(f'finetuned_{metric}_mean', 'mean'),
        fs=(f'finetuned_{metric}_std',  'mean'),
    ).reset_index()
    frac = grp['clean_frac'].values
    y    = np.where(np.isclose(frac, 0), grp['nm'].values, grp['fm'].values)
    yerr = np.where(np.isclose(frac, 0), grp['ns'].values, grp['fs'].values)
    return frac, y, yerr

## Plot 1 — Recovery curve (Spearman correlation)
Sensitive region Spearman correlation vs adversary clean fraction. One line per noise type, faceted by censor_split.  
Dotted reference line = clean (no-noise) baseline from original sweep.

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

n = len(CENSOR_SPLITS)
fig, axs = plt.subplots(1, n, figsize=(4.5 * n, 4), sharey=True, constrained_layout=True)
if n == 1:
    axs = [axs]

for i, split in enumerate(CENSOR_SPLITS):
    ax = axs[i]
    ax.set_title(f'{int(split * 100)}% sensitive data', fontsize=12)
    if i == 0:
        ax.set_ylabel('Spearman correlation\n(sensitive region)', fontsize=15)
    if i == 1:
        ax.set_xlabel('Fraction of clean data', fontsize=15)
    ax.set_ylim(-0.2, 0.8)
    ax.set_xlim(-0.03, 1.03)
    ax.tick_params(labelsize=12)

    curves = {}
    for nt in NOISE_TYPES:
        frac, y, yerr = recovery_curve(df, nt, split)
        ax.plot(frac, y, marker=NT_MARKERS[nt], label=LABELS[nt], color=NT_COLORS[nt])
        ax.fill_between(frac, y - yerr, y + yerr, alpha=0.2, color=NT_COLORS[nt])
        curves[nt] = (frac, y, yerr)
    ax.grid(True)

    if np.isclose(split, 0.9):
        axins = inset_axes(ax, width='45%', height='40%',
                           bbox_to_anchor=(0.0, 0.05, 1, 1),
                           bbox_transform=ax.transAxes,
                           loc='lower right')
        for nt in NOISE_TYPES:
            frac, y, yerr = curves[nt]
            mask = frac >= 0.2
            axins.plot(frac[mask], y[mask], marker=NT_MARKERS[nt], color=NT_COLORS[nt], markersize=4)
            axins.fill_between(frac[mask], y[mask] - yerr[mask], y[mask] + yerr[mask],
                               alpha=0.2, color=NT_COLORS[nt])
        axins.set_xlim(0.47, 1.02)
        axins.set_ylim(0.65, 0.73)
        axins.tick_params(labelsize=10)
        axins.grid(True)
        mark_inset(ax, axins, loc1=2, loc2=1, fc='none', ec='0.3', linewidth=1.0)

axs[0].legend(fontsize=10, loc='lower right')
plt.savefig('paper_figs/adversarial_recovery_corr.pdf', bbox_inches='tight')
plt.show()

## Plot 2 — Correlation vs noise level (rows: noise type, columns: % sensitive data)
Same axes as Plot 5 but fully faceted — no averaging across censor_splits.

In [ ]:
noise_type_order = ['omission', 'xnoise', 'ynoise']
n_rows = len(noise_type_order)
n_cols = len(CENSOR_SPLITS)

fig, axs = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4 * n_rows), sharey='row')

for r, nt in enumerate(noise_type_order):
    sub = df[df['noise_type'] == nt].copy()
    sub['noise_level'] = sub['noise_param'].apply(lambda p: noise_param_to_level(nt, p))

    for c, split in enumerate(CENSOR_SPLITS):
        ax = axs[r, c]
        if r == 0:
            ax.set_title(f'{int(split * 100)}% sensitive data', fontsize=11)
        if c == 0:
            ax.set_ylabel(f'{LABELS[nt]}\nSpearman r', fontsize=9)
        if r == n_rows - 1:
            ax.set_xlabel('Noise level', fontsize=9)

        ssub = sub[sub['censor_split'] == split]
        for k, frac in enumerate(sorted(ssub['clean_frac'].unique())):
            fsub = ssub[ssub['clean_frac'] == frac]
            grp = fsub.groupby('noise_level').agg(
                nm=('noised_upper_corr_mean',    'mean'),
                ns=('noised_upper_corr_std',     'mean'),
                fm=('finetuned_upper_corr_mean', 'mean'),
                fs=('finetuned_upper_corr_std',  'mean'),
            ).reset_index()

            levels = grp['noise_level'].values
            y    = np.where(np.isclose(frac, 0), grp['nm'].values, grp['fm'].values)
            yerr = np.where(np.isclose(frac, 0), grp['ns'].values, grp['fs'].values)

            ax.plot(levels, y, marker='o', label=f'clean_frac={frac}', color=f'C{k}')
            ax.fill_between(levels, y - yerr, y + yerr, alpha=0.15, color=f'C{k}')

        ax.grid(True)

# single legend outside
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, fontsize=8, loc='center right', bbox_to_anchor=(1.08, 0.5))
plt.tight_layout()
#plt.savefig('paper_figs/adversarial_corr_vs_noise_faceted.pdf', bbox_inches='tight')
plt.show()